In [ ]:
# System import
import sys
sys.path.insert(0, "..")

# Local import
from utils.data_processing import build_chroma_document_from_mongo_document
from utils.mongo_handler import get_legislation_by_query

In [ ]:
query = {"category": "Luật"}
result = get_legislation_by_query(query)
chroma_documents = [build_chroma_document_from_mongo_document(doc) for doc in result["data"]]
print(f"Number of documents: {len(chroma_documents)}")
print(f"First document: {chroma_documents[0]}")

In [ ]:
from langchain_core.documents import Document
docs = [Document(page_content=doc["documents"], metadata=doc["metadata"]) for doc in chroma_documents]

# Split the documents into smaller chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000,  # chunk size (characters)
    chunk_overlap=300,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)   

In [ ]:
import chromadb

client = chromadb.EphemeralClient()
collection = client.create_collection(
    name="legislation",
)

In [ ]:
def sanitize_metadata(metadata):
    """Sanitize metadata by removing keys that are not serializable."""
    for k, v in metadata.items():
        if isinstance(v, (dict, list)):
            metadata[k] = ", ".join(map(str, v))  # Convert to string
    return metadata

In [ ]:
import time

for doc in docs:
    chunked_docs = text_splitter.split_documents([doc])
    print(f"Number of chunks {doc.metadata['name']} splited into: {len(chunked_docs)}")
    
    for attempt in range(3):  # Try up to 3 times
        try:
            # Operation that might fail
            collection.add(
                documents=[doc.page_content for doc in chunked_docs],
                metadatas=[sanitize_metadata(doc.metadata) for doc in chunked_docs],
                ids=[(str(doc.metadata["id"]) + "_" + str(doc.metadata["start_index"])) for doc in chunked_docs],
            )
            break  # Exit the loop on success
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}. Retrying...")
            time.sleep(2)  # Wait before retrying
    else:
        print("Failed to add documents after 3 attempts. Skipping this document.")

In [ ]:
retrieve_result = collection.query(
    query_texts=["Các nguyên tắc cơ bản của hôn nhân và gia đình là gì?"],
    n_results=10,
)

for i, doc in enumerate(retrieve_result["documents"][0]):
    print(f"Document {i + 1}:")
    print("Name:", retrieve_result["metadatas"][0][i].get("name", "N/A"))
    print("Content:", doc)
    print("Id:", retrieve_result["ids"][0][i])
    print("Document number:", retrieve_result["metadatas"][0][i].get("numberDoc", "N/A"))
    print("Fields:", retrieve_result["metadatas"][0][i].get("fields", "N/A"))
    # print("Metadata:", retrieve_result["metadatas"][0][i])
    print("Score:", retrieve_result["distances"][0][i])
    print("-"* 50)  # Separator for readability

In [ ]:
client.delete_collection(name="legislation")
